# Pipeline — Tratamento de dados e features patrimoniais por candidato

Este notebook consolida a primeira fase do projeto: transformar a base bruta de bens declarados dos candidatos em uma base de **features patrimoniais por candidato**.

A unidade de entrada é um dataset de bens declarados pelos candidatos em uma eleição brasileira, esses podem ser encontrados no site do TSE(Tribunal Superior Eleitoral). Cada linha representa um bem declarado. A saída principal é o arquivo `features_patrimoniais_por_candidato.csv`, no qual cada linha representa um candidato com variáveis agregadas de patrimônio.

O pipeline foi organizado para ser **repetível e parametrizável**, com uma opção de auditoria em mais detalhes podendo ser ativada


## 0. Análise Exploratória da Base Original
Esta seção apresenta uma visão inicial da base de bens declarados antes de qualquer etapa de tratamento, agregação ou engenharia de variáveis. O objetivo é registrar a estrutura original dos dados, avaliar sua qualidade.

In [66]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

pd.set_option("display.max_columns", 100)
pd.set_option("display.float_format", lambda x: f"{x:,.2f}")

CAMINHO_BENS = "bem_candidato_2022_BRASIL.csv"

df_bens = pd.read_csv(
    CAMINHO_BENS,
    sep=";",
    encoding="latin1",
    dtype=str
)

print(f"Linhas: {df_bens_raw.shape[0]:,}")
print(f"Colunas: {df_bens_raw.shape[1]:,}")

display(df_bens.head())

Linhas: 92,538
Colunas: 19


,DT_GERACAO,HH_GERACAO,ANO_ELEICAO,CD_TIPO_ELEICAO,NM_TIPO_ELEICAO,CD_ELEICAO,DS_ELEICAO,DT_ELEICAO,SG_UF,SG_UE,NM_UE,SQ_CANDIDATO,NR_ORDEM_BEM_CANDIDATO,CD_TIPO_BEM_CANDIDATO,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO,DT_ULT_ATUAL_BEM_CANDIDATO,HH_ULT_ATUAL_BEM_CANDIDATO
0,04/05/2026,03:30:11,2022,2,Eleição Ordinária,546,Eleições Gerais Estaduais 2022,02/10/2022,AL,AL,ALAGOAS,20001652525,13,99,OUTROS BENS E DIREITOS,"PARTICIPACAO NA SOCIEDADE HOSPITAL SANTA INES,...","10000,00",02/10/2022,23:16:22
1,04/05/2026,03:30:11,2022,2,Eleição Ordinária,546,Eleições Gerais Estaduais 2022,02/10/2022,AL,AL,ALAGOAS,20001652525,14,63,Dinheiro em espécie - moeda nacional,DINHEIRO EM ESPECIE,"30000,00",02/10/2022,23:16:22
2,04/05/2026,03:30:11,2022,2,Eleição Ordinária,546,Eleições Gerais Estaduais 2022,02/10/2022,AL,AL,ALAGOAS,20001652525,15,99,OUTROS BENS E DIREITOS,CAMARA MUNICIPAL DE CORURIPE,"104000,00",02/10/2022,23:16:22
3,04/05/2026,03:30:11,2022,2,Eleição Ordinária,546,Eleições Gerais Estaduais 2022,02/10/2022,AL,AL,ALAGOAS,20001652526,1,99,OUTROS BENS E DIREITOS,RENDIMENTO ANUAL ALAGOAS CAMARA DOS VEREADORES,"66080,00",02/10/2022,23:16:22
4,04/05/2026,03:30:11,2022,2,Eleição Ordinária,546,Eleições Gerais Estaduais 2022,02/10/2022,AL,AL,ALAGOAS,20001652527,1,99,OUTROS BENS E DIREITOS,CONTA CORRENTE,"993,66",02/10/2022,23:16:22


In [67]:
resumo_base = pd.DataFrame({
    "indicador": [
        "Registros de bens declarados",
        "Candidatos únicos com bens declarados",
        "Tipos oficiais de bens",
            "UFs presentes",
            "Valores Duplicados"
        ],
        "valor": [
            len(df_bens),
            df_bens["SQ_CANDIDATO"].nunique(),
            df_bens["DS_TIPO_BEM_CANDIDATO"].nunique(),
            df_bens["SG_UF"].nunique(),
            df_bens.duplicated().sum()
        ]
    })

display(resumo_base)

,indicador,valor
0,Registros de bens declarados,92538
1,Candidatos únicos com bens declarados,18245
2,Tipos oficiais de bens,50
3,UFs presentes,28
4,Valores Duplicados,0


In [68]:
df_bens_diag = df_bens.copy()

df_bens_diag["VR_BEM_CANDIDATO_NUM"] = (
        df_bens_diag["VR_BEM_CANDIDATO"]
        .str.replace(".", "", regex=False)
        .str.replace(",", ".", regex=False)
        .astype(float)
    )

resumo_valores = pd.DataFrame({
        "indicador": [
            "Valor total declarado",
            "Valor médio por bem",
            "Valor mediano por bem",
            "Menor valor declarado",
            "Maior valor declarado",
            "Bens com valor zero",
            "Bens com valor negativo"
        ],
        "valor": [
            df_bens_diag["VR_BEM_CANDIDATO_NUM"].sum(),
            df_bens_diag["VR_BEM_CANDIDATO_NUM"].mean(),
            df_bens_diag["VR_BEM_CANDIDATO_NUM"].median(),
            df_bens_diag["VR_BEM_CANDIDATO_NUM"].min(),
            df_bens_diag["VR_BEM_CANDIDATO_NUM"].max(),
            (df_bens_diag["VR_BEM_CANDIDATO_NUM"] == 0).sum(),
            (df_bens_diag["VR_BEM_CANDIDATO_NUM"] < 0).sum()
        ]
    })

display(resumo_valores)

display(
    df_bens_diag["VR_BEM_CANDIDATO_NUM"]
    .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    .to_frame("valor_do_bem")
)

,indicador,valor
0,Valor total declarado,"23,681,228,242.17"
1,Valor médio por bem,"255,908.15"
2,Valor mediano por bem,"45,800.00"
3,Menor valor declarado,0.00
4,Maior valor declarado,"1,083,531,773.61"
5,Bens com valor zero,410.00
6,Bens com valor negativo,0.00


,valor_do_bem
count,"92,538.00"
mean,"255,908.15"
std,"4,813,511.88"
min,0.00
1%,1.00
5%,148.46
25%,"9,835.54"
50%,"45,800.00"
75%,"150,000.00"
95%,"700,000.00"


In [69]:
tipos_bens = (
        df_bens_diag
        .groupby("DS_TIPO_BEM_CANDIDATO")
        .agg(
            qtd_registros=("SQ_CANDIDATO", "size"),
            qtd_candidatos=("SQ_CANDIDATO", "nunique"),
            valor_total=("VR_BEM_CANDIDATO_NUM", "sum"),
            valor_mediano=("VR_BEM_CANDIDATO_NUM", "median")
        )
        .reset_index()
    )

tipos_bens["pct_registros"] = tipos_bens["qtd_registros"] / len(df_bens_diag) * 100
tipos_bens["pct_valor_total"] = tipos_bens["valor_total"] / df_bens_diag["VR_BEM_CANDIDATO_NUM"].sum() * 100

print("Tipos de bens mais frequentes:")
display(
        tipos_bens
        .sort_values("qtd_registros", ascending=False)
        .head(15)
    )

print("Tipos de bens com maior valor total declarado:")
display(
        tipos_bens
        .sort_values("valor_total", ascending=False)
        .head(15)
    )

Tipos de bens mais frequentes:


,DS_TIPO_BEM_CANDIDATO,qtd_registros,qtd_candidatos,valor_total,valor_mediano,pct_registros,pct_valor_total
49,"Veículo automotor terrestre: caminhão, automóv...",17313,11198,"1,132,159,016.51","41,000.00",18.71,4.78
46,Terreno,9179,4200,"1,829,224,956.59","52,000.00",9.92,7.72
7,Casa,8858,6677,"2,841,591,654.29","191,029.85",9.57,12.00
12,Depósito bancário em conta corrente no País,8508,4895,"294,074,647.88","2,742.70",9.19,1.24
1,Apartamento,6869,4603,"2,348,562,335.57","210,000.00",7.42,9.92
43,Quotas ou quinhões de capital,6094,3091,"3,512,235,949.13","30,000.00",6.59,14.83
30,OUTROS BENS E DIREITOS,4689,2018,"1,875,920,183.66","36,000.00",5.07,7.92
34,Outros bens imóveis,4373,2119,"1,567,205,284.26","100,000.00",4.73,6.62
2,"Aplicação de renda fixa (CDB, RDB e outros)",4202,2383,"738,146,574.40","15,000.00",4.54,3.12
6,Caderneta de poupança,3250,2355,"95,553,197.41","2,487.56",3.51,0.40


Tipos de bens com maior valor total declarado:


,DS_TIPO_BEM_CANDIDATO,qtd_registros,qtd_candidatos,valor_total,valor_mediano,pct_registros,pct_valor_total
43,Quotas ou quinhões de capital,6094,3091,"3,512,235,949.13","30,000.00",6.59,14.83
7,Casa,8858,6677,"2,841,591,654.29","191,029.85",9.57,12.00
1,Apartamento,6869,4603,"2,348,562,335.57","210,000.00",7.42,9.92
33,Outras participações societárias,1589,991,"2,341,014,827.60","40,000.00",1.72,9.89
30,OUTROS BENS E DIREITOS,4689,2018,"1,875,920,183.66","36,000.00",5.07,7.92
46,Terreno,9179,4200,"1,829,224,956.59","52,000.00",9.92,7.72
34,Outros bens imóveis,4373,2119,"1,567,205,284.26","100,000.00",4.73,6.62
49,"Veículo automotor terrestre: caminhão, automóv...",17313,11198,"1,132,159,016.51","41,000.00",18.71,4.78
3,Ações (inclusive as provenientes de linha tele...,2377,712,"1,006,192,652.29","7,243.56",2.57,4.25
11,Crédito decorrente de empréstimo,611,365,"757,658,045.30","180,583.75",0.66,3.20


In [70]:
mask_outros = (
        df_bens_diag["DS_TIPO_BEM_CANDIDATO"]
        .str.upper()
        .str.contains("OUTROS BENS", na=False)
    )

resumo_outros = pd.DataFrame({
        "indicador": [
            "Registros em Outros bens e direitos",
            "% dos registros em Outros bens e direitos",
            "Candidatos com ao menos um bem em Outros",
            "% dos candidatos com ao menos um bem em Outros",
            "Valor total em Outros bens e direitos",
            "% do valor total em Outros bens e direitos"
        ],
        "valor": [
            mask_outros.sum(),
            mask_outros.mean() * 100,
            df_bens_diag.loc[mask_outros, "SQ_CANDIDATO"].nunique(),
            df_bens_diag.loc[mask_outros, "SQ_CANDIDATO"].nunique() / df_bens_diag["SQ_CANDIDATO"].nunique() * 100,
            df_bens_diag.loc[mask_outros, "VR_BEM_CANDIDATO_NUM"].sum(),
            df_bens_diag.loc[mask_outros, "VR_BEM_CANDIDATO_NUM"].sum() / df_bens_diag["VR_BEM_CANDIDATO_NUM"].sum() * 100
        ]
    })

display(resumo_outros)

base_candidato_diag = (
        df_bens_diag
        .groupby("SQ_CANDIDATO")
        .agg(
            qtd_bens=("NR_ORDEM_BEM_CANDIDATO", "count"),
            patrimonio_total=("VR_BEM_CANDIDATO_NUM", "sum"),
            qtd_tipos_bem=("DS_TIPO_BEM_CANDIDATO", "nunique"),
            uf=("SG_UF", "first")
        )
        .reset_index()
    )

display(
        base_candidato_diag[["qtd_bens", "patrimonio_total", "qtd_tipos_bem"]]
        .describe(percentiles=[0.01, 0.05, 0.25, 0.50, 0.75, 0.95, 0.99])
    )

display(
        base_candidato_diag
        .sort_values("patrimonio_total", ascending=False)
        .head(10)
    )

,indicador,valor
0,Registros em Outros bens e direitos,"10,060.00"
1,% dos registros em Outros bens e direitos,10.87
2,Candidatos com ao menos um bem em Outros,"4,099.00"
3,% dos candidatos com ao menos um bem em Outros,22.47
4,Valor total em Outros bens e direitos,"3,649,079,231.17"
5,% do valor total em Outros bens e direitos,15.41


,qtd_bens,patrimonio_total,qtd_tipos_bem
count,"18,245.00","18,245.00","18,245.00"
mean,5.07,"1,297,957.15",3.03
std,7.12,"13,683,167.83",2.35
min,1.00,0.00,1.00
1%,1.00,216.63,1.00
5%,1.00,"7,000.00",1.00
25%,1.00,"77,543.60",1.00
50%,3.00,"276,000.00",2.00
75%,6.00,"757,637.48",4.00
95%,16.00,"3,259,536.88",8.00


,SQ_CANDIDATO,qtd_bens,patrimonio_total,qtd_tipos_bem,uf
17639,90001615125,58,"1,267,950,846.18",16,GO
16937,70001723172,91,"618,868,229.48",11,DF
16674,70001621520,28,"453,595,515.70",13,DF
3458,140001611624,7,"448,447,364.52",7,PA
10149,240001605916,6,"390,033,568.89",6,SC
1120,110001643542,49,"378,869,597.56",13,MT
8651,210001609945,7,"372,943,176.46",6,RS
9410,220001614362,70,"351,588,736.22",9,RO
1945,130001605813,6,"178,215,029.38",5,MG
16260,60001643110,128,"158,184,458.79",10,CE


## 1. Parâmetros da geração

Altere esta célula antes de cada execução. Os parâmetros mais importantes para iteração são `NOME_RODADA`, `CAMINHO_BENS_UNIFICADO`, `AJUSTES_TIPO_MACRO`, `REGRAS_MANUAIS_TEXTO` e a seleção de `ARQUIVOS_SAIDA`.

Por padrão, todos os arquivos finais são gerados. Para não gerar algum deles, troque o respectivo valor em `ARQUIVOS_SAIDA` para `False`.


In [71]:
from pathlib import Path
import csv
import json
import re
import shutil
import unicodedata
from datetime import datetime

import numpy as np
import pandas as pd

# ============================================================
# Parâmetros principais da rodada
# ============================================================

PARAMS = {
    # Identificação da rodada. Troque este valor quando quiser uma nova execução sem sobrescrever saídas.
    "NOME_RODADA": "features_patrimoniais_por_candidato",

    # Caminho de entrada.
    "CAMINHO_BENS_UNIFICADO": "bem_candidato_2022_BRASIL.csv",

    # Saídas.
    "DIR_SAIDA_BASE": ".",

    # Leitura/escrita.
    # Use "auto" para detectar encoding/separador quando necessário.
    "SEP_ENTRADA": ";",
    "ENCODING_ENTRADA": "latin1",
    "DECIMAL_ENTRADA": ",",
    "SEP_SAIDA": ";",
    "ENCODING_SAIDA": "utf-8-sig",

    # Tratamento de texto.
    "CORRIGIR_MOJIBAKE": True,

    # Tratamento.
    "REMOVER_DUPLICADOS": True,
    "REMOVER_CANDIDATOS_PATRIMONIO_ZERO_FEATURES": True,
    "VALOR_PREENCHER_MACRO_NA": "outros",

    # Macros fixas esperadas no dataset final.
    # A ordem abaixo define sempre a ordem das colunas de saída.
    "MACROS_FEATURES_PADRAO": [
        "ativos_financeiros",
        "bens_luxo_colecao",
        "creditos_direitos",
        "dinheiro_especie",
        "direitos_intangiveis",
        "imoveis",
        "outros",
        "outros_atividade_profissional",
        "participacoes_societarias",
        "rural_agropecuario",
        "veiculos",
    ],

    # Auditoria opcional.
    # Por padrão, o pipeline só gera o dataset final de features.
    # Quando True, cria uma pasta de auditoria da rodada com a base de bens
    "GERAR_AUDITORIA": False,

    # Exportação auxiliar.
    "GERAR_CONFIG": True,
    "GERAR_ZIP_RODADA": False,
}

# ============================================================
# Ajustes manuais por tipo oficial do TSE
# ============================================================
# Use quando um valor específico de DS_TIPO_BEM_CANDIDATO precisar ser forçado
# para uma macro categoria.

AJUSTES_TIPO_MACRO = {}

# ============================================================
# Regras manuais por texto
# ============================================================
# Cada regra é aplicada sobre texto normalizado formado por:
# DS_TIPO_BEM_CANDIDATO + DS_BEM_CANDIDATO.
#
# Campos:
# - nome: identificador da regra
# - padrao: regex em texto normalizado
# - macro: macro_categoria_refinada desejada
# - submacro_col: coluna de submacro a preencher opcionalmente
# - submacro_valor: valor opcional da submacro

REGRAS_MANUAIS_TEXTO = []


## 2. Mapeamento base de tipos de bens para macro categorias

Esta é a classificação inicial por tipo oficial de bem. As funções seguintes refinam casos genéricos ou ambíguos a partir da descrição textual.

In [72]:
MAPA_TIPO_MACRO_BASE = {
    # Imóveis
    "Apartamento": "imoveis",
    "Casa": "imoveis",
    "Terreno": "imoveis",
    "Sala ou conjunto": "imoveis",
    "Galpão": "imoveis",
    "Loja": "imoveis",
    "Prédio comercial": "imoveis",
    "Prédio residencial": "imoveis",
    "Construção": "imoveis",
    "Benfeitorias": "imoveis",
    "Outros bens imóveis": "imoveis",

    # Rural / agropecuário
    "Terra nua": "rural_agropecuario",

    # Veículos
    "Veículo automotor terrestre: caminhão, automóvel, moto, etc.": "veiculos",
    "Embarcação": "veiculos",
    "Aeronave": "veiculos",

    # Ativos financeiros
    "Aplicação de renda fixa (CDB, RDB e outros)": "ativos_financeiros",
    "Caderneta de poupança": "ativos_financeiros",
    "Depósito bancário em conta corrente no País": "ativos_financeiros",
    "Depósito bancário em conta corrente no exterior": "ativos_financeiros",
    "VGBL - Vida Gerador de Benefício Livre": "ativos_financeiros",
    "Ações (inclusive as provenientes de linha telefônica)": "ativos_financeiros",
    "Fundo de Investimento Imobiliário": "ativos_financeiros",
    "Outros fundos": "ativos_financeiros",
    "Fundo de Curto Prazo": "ativos_financeiros",
    "Fundo de Longo Prazo e Fundo de Investimentos em Direitos Creditórios (FIDC)": "ativos_financeiros",
    "Fundos: Ações, Mútuos de Privatização, Invest. Empresas Emergentes, Invest.Participação e Invest. Índice Mercado": "ativos_financeiros",
    "Ouro, ativo financeiro": "ativos_financeiros",
    "Mercado futuros, de opções e a termo": "ativos_financeiros",
    "Plano PAIT e caderneta de pecúlio": "ativos_financeiros",
    "Outras aplicações e Investimentos": "ativos_financeiros",
    "Outros depósitos à vista e numerário": "ativos_financeiros",

    # Participações societárias
    "Quotas ou quinhões de capital": "participacoes_societarias",
    "Outras participações societárias": "participacoes_societarias",

    # Dinheiro
    "Dinheiro em espécie - moeda nacional": "dinheiro_especie",
    "Dinheiro em espécie - moeda estrangeira": "dinheiro_especie",

    # Créditos e direitos
    "Crédito decorrente de alienação": "creditos_direitos",
    "Crédito decorrente de empréstimo": "creditos_direitos",
    "Outros créditos e poupança vinculados": "creditos_direitos",
    "Consórcio não contemplado": "creditos_direitos",
    "Poupança para construção ou aquisição de bem imóvel": "creditos_direitos",
    "Leasing": "creditos_direitos",

    # Bens de luxo / coleção
    "Jóia, quadro, objeto de arte, de coleção, antiguidade, etc.": "bens_luxo_colecao",

    # Direitos intangíveis
    "Direito de autor, de inventor e patente": "direitos_intangiveis",
    "Direito de lavra e assemelhado": "direitos_intangiveis",
    "Licença e concessões especiais": "direitos_intangiveis",

    # Outros
    "Linha telefônica": "outros",
    "Título de clube e assemelhado": "outros",
    "Bem relacionado com o exercício da atividade autônoma": "outros",
    "Outros bens móveis": "outros",
    "OUTROS BENS E DIREITOS": "outros",
}

MAPA_TIPO_MACRO = {**MAPA_TIPO_MACRO_BASE, **AJUSTES_TIPO_MACRO}

## 3. Leitura robusta, correção de encoding e utilitários

Esta seção corrige os problemas de formatação textual observados (`Ã£`, `Ã©`, `ï»¿`) e detecta automaticamente separador/encoding.

In [73]:
def parece_mojibake(texto):
    """Identifica padrões comuns de texto lido com encoding errado."""
    if not isinstance(texto, str):
        return False
    marcadores = ("Ã", "Â", "â€", "â€“", "â€œ", "â€�", "ï»¿", "�")
    return any(m in texto for m in marcadores)


def remover_bom_colunas(df):
    """Remove BOM e espaços residuais dos nomes das colunas."""
    df = df.copy()
    df.columns = [str(c).replace("\ufeff", "").replace("ï»¿", "").strip() for c in df.columns]
    return df


def corrigir_mojibake_texto(x):
    """
    Corrige mojibake em textos como 'SÃ£o', 'AplicaÃ§Ã£o' e 'ï»¿'.
    A correção só é aplicada quando há marcadores claros de encoding errado.
    """
    if pd.isna(x):
        return x
    if not isinstance(x, str):
        return x

    s = x.replace("\ufeff", "").replace("ï»¿", "")

    for _ in range(3):
        if not parece_mojibake(s):
            break
        try:
            novo = s.encode("latin1", errors="strict").decode("utf-8", errors="strict")
        except Exception:
            break
        if novo == s:
            break
        s = novo.replace("\ufeff", "").replace("ï»¿", "")

    return s


def corrigir_mojibake_dataframe(df, params):
    """Aplica correção de mojibake em colunas textuais e nomes de colunas."""
    df = remover_bom_colunas(df)
    if not params.get("CORRIGIR_MOJIBAKE", True):
        return df

    df = df.copy()
    df.columns = [corrigir_mojibake_texto(c) for c in df.columns]

    colunas_texto = df.select_dtypes(include=["object", "string"]).columns
    for col in colunas_texto:
        df[col] = df[col].map(corrigir_mojibake_texto)

    return df


def detectar_sep(caminho, encoding):
    """Detecta separador entre ';' e ',' usando a primeira linha decodificada."""
    with open(caminho, "rb") as f:
        amostra = f.read(4096)
    texto = amostra.decode(encoding, errors="replace")
    primeira_linha = texto.splitlines()[0] if texto.splitlines() else ""
    if primeira_linha.count(";") >= primeira_linha.count(","):
        return ";"
    return ","


def score_mojibake_df(df):
    """Pontua a presença de mojibake em uma amostra de dataframe."""
    score = 0
    for col in df.columns:
        score += 5 if parece_mojibake(str(col)) else 0
    colunas_texto = df.select_dtypes(include=["object", "string"]).columns[:6]
    for col in colunas_texto:
        amostra = df[col].dropna().astype(str).head(200)
        score += sum(1 for x in amostra if parece_mojibake(x))
    return score


def ler_csv_robusto(caminho, params):
    """
    Lê CSV tentando evitar dois problemas comuns do projeto:
    1. Separador diferente entre bases intermediárias e arquivos brutos.
    2. Encoding incorreto, que gera textos como 'SÃ£o' ou 'AplicaÃ§Ã£o'.
    """
    caminho = Path(caminho)
    enc_param = params.get("ENCODING_ENTRADA", "auto")
    sep_param = params.get("SEP_ENTRADA", "auto")
    decimal_param = params.get("DECIMAL_ENTRADA", None)
    read_kwargs = {} if decimal_param is None else {"decimal": decimal_param}
    if str(caminho).endswith("bem_candidato_2022_BRASIL.csv"):
        read_kwargs["dtype"] = {"SQ_CANDIDATO": "Int64"}

    encodings = ["utf-8-sig", "utf-8", "cp1252", "latin1"] if enc_param == "auto" else [enc_param]
    candidatos = []
    erros = []

    for enc in encodings:
        try:
            sep = detectar_sep(caminho, enc) if sep_param == "auto" else sep_param
            df_amostra = pd.read_csv(caminho, sep=sep, encoding=enc, nrows=300, low_memory=False, **read_kwargs)
            score = score_mojibake_df(df_amostra)
            candidatos.append((score, enc, sep))
        except Exception as e:
            erros.append((enc, str(e)))

    if not candidatos:
        raise ValueError(f"Não foi possível ler {caminho}. Erros: {erros}")

    candidatos = sorted(candidatos, key=lambda x: x[0])
    _, encoding_escolhido, sep_escolhido = candidatos[0]

    df = pd.read_csv(caminho, sep=sep_escolhido, encoding=encoding_escolhido, low_memory=False, **read_kwargs)
    df = corrigir_mojibake_dataframe(df, params)
    return df, {"encoding": encoding_escolhido, "sep": sep_escolhido}


def normalizar_texto(x):
    """Normaliza texto para uso em regras baseadas em regex."""
    if pd.isna(x):
        return ""

    x = corrigir_mojibake_texto(x)
    x = str(x).lower()
    x = unicodedata.normalize("NFKD", x)
    x = "".join(c for c in x if not unicodedata.combining(c))
    x = re.sub(r"[^a-z0-9\s]", " ", x)
    x = re.sub(r"\s+", " ", x).strip()
    return x


def normalizar_valor_monetario(serie, params=None):
    """
    Converte valores monetários do TSE para numérico.

    Trata:
    - '10000,00' -> 10000.00
    - '10.000,00' -> 10000.00
    - '993,66' -> 993.66
    - 10000.0 -> 10000.00
    - '10000.0' -> 10000.00
    - '1.267.950.846,18' -> 1267950846.18
    """

    if pd.api.types.is_numeric_dtype(serie):
        return pd.to_numeric(serie, errors="coerce").fillna(0)

    s = serie.astype(str).str.strip()

    # Remove caracteres invisíveis comuns
    s = (
        s
        .str.replace("\ufeff", "", regex=False)
        .str.replace("ï»¿", "", regex=False)
        .str.replace("\xa0", "", regex=False)
        .str.replace(" ", "", regex=False)
    )

    # Mantém apenas dígitos, ponto, vírgula e sinal negativo
    s = s.str.replace(r"[^0-9,\.\-]", "", regex=True)

    def converter_valor(x):
        if x in {"", "nan", "NaN", "None"}:
            return 0.0

        # Caso brasileiro: tem vírgula decimal
        if "," in x:
            x = x.replace(".", "").replace(",", ".")
            return float(x) if x not in {"", "-", "."} else 0.0

        # Caso internacional ou inteiro puro
        return float(x) if x not in {"", "-", "."} else 0.0

    return s.map(converter_valor).fillna(0)


def localizar_coluna(df, candidatos):
    """Retorna a primeira coluna existente dentro de uma lista de nomes candidatos."""
    for col in candidatos:
        if col in df.columns:
            return col
    return None


def resumo_categoria(df, categoria_col="macro_categoria_refinada", valor_col="VR_BEM_CANDIDATO_NUM"):
    """Gera resumo por categoria, com contagem e estatísticas de valor."""
    if df.empty or categoria_col not in df.columns:
        return pd.DataFrame()

    out = (
        df.groupby(categoria_col)[valor_col]
        .agg(qtd="count", valor_total="sum", valor_mediano="median", valor_medio="mean")
        .sort_values("valor_total", ascending=False)
    )

    total_valor = out["valor_total"].sum()
    total_qtd = out["qtd"].sum()

    out["perc_valor_total"] = np.where(total_valor > 0, out["valor_total"] / total_valor * 100, 0).round(2)
    out["perc_qtd"] = np.where(total_qtd > 0, out["qtd"] / total_qtd * 100, 0).round(2)

    return out


def ler_base_bens(params):
    """Lê sempre a base unificada informada em CAMINHO_BENS_UNIFICADO."""
    caminho_unificado = Path(params["CAMINHO_BENS_UNIFICADO"])

    if not caminho_unificado.exists():
        raise FileNotFoundError(
            f"Não encontrei o arquivo unificado: {caminho_unificado}. "
            "Ajuste CAMINHO_BENS_UNIFICADO para apontar para o CSV de entrada."
        )

    df, info = ler_csv_robusto(caminho_unificado, params)
    origem = f"arquivo_unificado:{caminho_unificado} | encoding={info['encoding']} | sep={info['sep']}"
    return df, origem


def preparar_base_inicial(df, params):
    """Padroniza colunas essenciais e cria campos auxiliares."""
    df = corrigir_mojibake_dataframe(df, params)
    df = df.copy()

    if "VR_BEM_CANDIDATO" in df.columns:
        col_valor = "VR_BEM_CANDIDATO"
    elif "VR_BEM_CANDIDATO_NUM" in df.columns:
        col_valor = "VR_BEM_CANDIDATO_NUM"
    else:
        raise KeyError("A base precisa conter VR_BEM_CANDIDATO ou VR_BEM_CANDIDATO_NUM.")

    if "VR_BEM_CANDIDATO" not in df.columns:
        df["VR_BEM_CANDIDATO"] = df[col_valor]

    df["VR_BEM_CANDIDATO_NUM"] = normalizar_valor_monetario(df[col_valor], params)

    for col in ["DS_TIPO_BEM_CANDIDATO", "DS_BEM_CANDIDATO"]:
        if col not in df.columns:
            df[col] = ""

    df["tipo_norm"] = df["DS_TIPO_BEM_CANDIDATO"].apply(normalizar_texto)
    df["desc_norm"] = df["DS_BEM_CANDIDATO"].apply(normalizar_texto)
    df["texto_norm"] = (df["tipo_norm"].astype(str) + " " + df["desc_norm"].astype(str)).str.strip()

    df["macro_categoria"] = df["DS_TIPO_BEM_CANDIDATO"].map(MAPA_TIPO_MACRO)
    df["macro_categoria"] = df["macro_categoria"].fillna(params["VALOR_PREENCHER_MACRO_NA"])
    df["macro_categoria_refinada"] = df["macro_categoria"]

    dist = df["macro_categoria"].value_counts(normalize=True) * 100
    df["perc_registros"] = df["macro_categoria"].map(dist).round(2)

    if params["REMOVER_DUPLICADOS"]:
        subset_dup = [
            col for col in [
                "SQ_CANDIDATO",
                "NR_ORDEM_BEM_CANDIDATO",
                "DS_TIPO_BEM_CANDIDATO",
                "DS_BEM_CANDIDATO",
                "VR_BEM_CANDIDATO_NUM"
            ]
            if col in df.columns
        ]
        if subset_dup:
            df = df.drop_duplicates(subset=subset_dup).copy()
        else:
            df = df.drop_duplicates().copy()

    return df


## 4. Regras  de refinamento

As funções abaixo servem para redistribuir valores frequentes porém que foram encontrados em uma macro incorreta, o processo foi feito de forma iterativa apos varias rodadas de auditoria e registro dos principais casos incorretos.

In [74]:
def classificar_submacro_imoveis(row):
    texto = f"{row['tipo_norm']} {row['desc_norm']}"

    padrao_participacao = (
        r"\b("
        r"participacao|participacoes|part|sociedade|socied|"
        r"capital social|quotas|quinhao|quinhoes|empreend|"
        r"empreendimento|empresa|ltda|limitada|cnpj"
        r")\b"
    )
    padrao_rural = (
        r"\b("
        r"rural|fazenda|faz|sitio|sitios|chacara|chacaras|"
        r"gleba|terra nua|propriedade rural|zona rural|area rural|"
        r"imovel rural|terras rurais|estancia|rancho|granja|"
        r"haras|aras|hectare|hectares"
        r")\b"
        r"|(\bha\b|\bhas\b)|ha de terras|has de terras"
    )
    padrao_comercial = r"\b(comercial|sala|salas|loja|lojas|galpao|galpoes|barracao|predio comercial|ponto comercial|shopping|empresarial|office|industrial|comercio|padaria|pousada|supermercado|hospital|hotel|posto|clinica|restaurante|escritorio)\b"
    padrao_construcao = r"\b(construcao|construcoes|benfeitoria|benfeitorias|obra|obras|andamento|reforma|edificacao|edificacoes|planta)\b"
    padrao_residencial = r"\b(casa|casas|apartamento|apartamentos|apto|apt|residencial|residencia|condominio|edificio|sobrado|alvenaria|moradia)\b"
    padrao_terreno = r"\b(terreno|terrenos|lote|lotes|quadra|loteamento|fracao|parte ideal)\b"
    padrao_urbano_generico = r"\b(urbano|imovel urbano|cidade|municipio|rua|avenida|bairro|distrito federal|sao paulo|porto alegre|miami|balneario)\b"

    if re.search(padrao_participacao, texto):
        return "participacoes_societarias"
    if re.search(padrao_rural, texto):
        return "imovel_rural"
    if re.search(padrao_comercial, texto):
        return "imovel_comercial_industrial"
    if re.search(padrao_construcao, texto):
        return "construcao_benfeitoria"
    if re.search(padrao_residencial, texto):
        return "imovel_residencial"
    if re.search(padrao_terreno, texto):
        return "terreno_lote"
    if re.search(padrao_urbano_generico, texto):
        return "imovel_urbano_generico"
    return "imovel_generico"


def classificar_submacro_financeira(row):
    tipo = row["tipo_norm"]
    desc = row["desc_norm"]
    texto = f"{tipo} {desc}"

    padrao_fundo_forte = r"\b(fundo|fundos|fii|fidc|fip|fic|fundo de investimento|fundos de investimento|fundo imobiliario|investimento imobiliario|multimercado|firf|fima|fim|cotas de fundos|cotas do fundo|cota de fundo|fi em cotas|fundo itau|fundo bradesco|fundo caixa)\b"
    padrao_participacao_inequivoca = r"\b(capital social da empresa|capital social de empresa|capital social|participacao societaria|participacoes societarias|participacao no capital social|participacao em empresa|participacao na empresa|capital da empresa|quota de capital social|quotas de capital social|cota de capital social|cotas de capital social|aporte de capital|capital em empresa)\b"
    padrao_rural = r"\b(gado|bovino|bovina|rebanho|semovente|boi|vaca|bezerro|novilha|animais do tipo bovino)\b"
    padrao_cripto = r"\b(bitcoin|btc|ethereum|cripto|criptoativo|criptomoeda|binance|moeda eletronica|carteira digital)\b"
    padrao_previdencia = r"\b(vgbl|pgbl|previdencia|vida gerador de beneficio|brasil prev|brasilprev|fic prev|peculio|pait|previ)\b"
    padrao_derivativos = r"\b(mercado futuro|mercado futuros|opcoes|termo|derivativo|derivativos)\b"
    padrao_ouro = r"\b(ouro|ativo financeiro ouro|ourocap|brasilcap)\b"
    padrao_fundos = r"\b(fundo|fundos|fii|fidc|fip|fic|fundo de investimento|multimercado|exec premium)\b"
    padrao_renda_fixa = r"\b(cdb|rdb|renda fixa|tesouro|selic|lci|lca|debenture|debentures|cri|cra|certificado de deposito|rf|di|cash)\b"
    padrao_acoes = r"\b(acoes|acao|ordinarias|preferenciais|b3|bolsa de valores|clear|xp investimentos|rico|modal)\b"
    padrao_poupanca = r"\b(poupanca|caderneta de poupanca)\b"
    padrao_conta_corrente = r"\b(conta corrente|deposito bancario|saldo em conta|conta bancaria|banco do brasil|bradesco|itau|santander|caixa economica|nubank|inter|agencia|c c|saldo bancario|banco sicoob|cef|disponibilidade financeira|numerario)\b"

    if re.search(padrao_fundo_forte, texto) and not re.search(padrao_participacao_inequivoca, texto):
        return "fundos_investimento"
    if re.search(padrao_participacao_inequivoca, texto):
        return "participacoes_societarias"
    if re.search(padrao_rural, texto):
        return "rural_agropecuario"
    if re.search(padrao_cripto, texto):
        return "criptoativos"
    if re.search(padrao_previdencia, texto):
        return "previdencia_vgbl"
    if re.search(padrao_derivativos, texto):
        return "mercado_derivativos"
    if re.search(padrao_ouro, texto):
        return "ouro_ativo_financeiro"
    if re.search(padrao_fundos, texto):
        return "fundos_investimento"
    if re.search(padrao_renda_fixa, texto):
        return "renda_fixa"
    if re.search(padrao_acoes, texto):
        return "acoes"
    if re.search(padrao_poupanca, texto):
        return "poupanca"
    if re.search(padrao_conta_corrente, texto):
        return "deposito_conta_corrente"
    if "aplicacao" in texto or "aplicacoes" in texto or "investimento" in texto:
        return "outros_investimentos_financeiros"
    return "financeiro_generico"


def classificar_submacro_outros(row):
    texto = f"{row['tipo_norm']} {row['desc_norm']}"

    padrao_participacao = r"\b(quotas ou quinhoes de capital|quota de capital|quotas de capital|cota de capital|cotas de capital|capital social|participacao societaria|participacoes societarias|participacao em empresa|participacao na empresa|capital da empresa|empresa .* ltda|ltda|cnpj|acoes ordinarias|acoes preferenciais)\b"
    padrao_creditos = r"\b(direitos creditorios|direito creditorio|credito|creditos|saldo a receber|valor a receber|a receber|mutuo|emprestimo|acao judicial|processo judicial|deposito judicial|judicial|precatorio|indenizacao|devolucao de imposto|imposto de renda|restituicao|dividendos a receber|heranca|direitos hereditarios|inventario|espolio)\b"
    padrao_financeiro = r"\b(conta corrente|saldo em conta|banco do brasil|banco|bradesco|itau|santander|caixa economica|cef|poupanca|aplicacao|aplicacoes|investimento|vgbl|pgbl|previdencia|fundo|fundos|cdb|rdb|renda fixa|tesouro|bitcoin|cripto|moeda corrente|dinheiro em especie|moeda nacional)\b"
    padrao_rural = r"\b(fazenda|sitio|chacara|gleba|area rural|imovel rural|propriedade rural|hectare|hectares|cabecas de gado|cabeca de gado|cabecas|cabeca|gado|bovino|bovinos|bovina|bovinas|rebanho|semovente|semoventes|boi|bois|vaca|vacas|bezerro|bezerros|novilha|novilhas|pecuaria|criacao de animais|animais bovinos|animais|implementos agricolas|implemento agricola|atividade rural|exploracao de atividade rural|terra rural|terras rurais)\b|(\bha\b|\bhas\b)"
    padrao_imoveis = r"\b(imovel|imoveis|apartamento|casa|terreno|lote|predio|sala comercial|loja|galpao|construcao|benfeitoria|area urbana|predio comercial|imovel comercial|vinicula|pousada|hotel|supermercado|padaria)\b"
    padrao_veiculos = r"\b(veiculo|automovel|carro|moto|motocicleta|caminhao|camionete|camioneta|onibus|renavam|placa|trator|lancha|barco|embarcacao|aeronave|aviao)\b"
    padrao_atividade_profissional = r"\b(atividade autonoma|exercicio da atividade autonoma|equipamento profissional|consultorio|escritorio|maquinas e equipamentos|ferramentas)\b"

    if re.search(padrao_participacao, texto):
        return "participacoes_societarias"
    if re.search(padrao_creditos, texto):
        return "creditos_direitos"
    if re.search(padrao_rural, texto):
        return "rural_agropecuario"
    if re.search(padrao_imoveis, texto):
        return "imoveis"
    if re.search(padrao_veiculos, texto):
        return "veiculos"
    if re.search(padrao_financeiro, texto):
        return "ativos_financeiros"
    if re.search(padrao_atividade_profissional, texto):
        return "outros_atividade_profissional"
    return "outros"


def classificar_submacro_veiculos(row):
    tipo = str(row["tipo_norm"])
    desc = str(row["desc_norm"])
    texto = desc

    if "aeronave" in tipo:
        return "aeronave"
    if "embarcacao" in tipo:
        return "embarcacao"

    padrao_maquina_agricola = r"\b(?:trator|tratores|colheitadeira|plantadeira|pulverizador|maquina agricola|maquinas agricolas|implemento agricola|implementos agricolas|grade aradora|arado|rocadeira|carreta agricola|plataforma de corte)\b"
    padrao_aeronave = r"\b(?:aeronave|aviao|helicoptero|ultraleve|embraer|phenom|aircraft|cessna|beech|prefixo pt|prefixo pr)\b"
    padrao_embarcacao = r"\b(?:barco|lancha|embarcacao|velereiro|veleiro|jetski|jet ski|iate|bote)\b"
    padrao_caminhao_onibus = r"\b(?:caminhao|caminhoes|onibus|microonibus|micro onibus|carreta|reboque|semi reboque|cavalinho|sprinter|ducato|kombi|furgon|furgao|volvo fh|vol vo|fh 460)\b"
    padrao_moto = r"\b(?:moto|motocicleta|motociclo|honda biz|biz|cg 150|cg150|yamaha|suzuki)\b"
    padrao_veiculo_leve = r"\b(?:carro|automovel|veiculo|veiuculo|veiuclo|camionete|camioneta|caminhonete|pickup|pick up|fiat|ford|chevrolet|toyota|honda|hyundai|hyndai|hyunday|volkswagen|vw|renault|peugeot|citroen|nissan|gm|jeep|mitsubishi|mercedes|bmw|audi|kia|volvo|land rover|range rover|range rovery|dodge|ram|amarok|hb20|seat|cordoba|tucson|hillux|hilux|gol|palio|uno|corolla|s10|ranger|civic|pajero|discovery|compass|sw4|onix|prisma|sandero|logan|ecosport|fox|saveiro|strada|toro|spin|cruze|ka|opala)\b"

    if re.search(padrao_maquina_agricola, texto):
        return "maquina_agricola"
    if re.search(padrao_aeronave, texto):
        return "aeronave"
    if re.search(padrao_embarcacao, texto):
        return "embarcacao"
    if re.search(padrao_caminhao_onibus, texto):
        return "caminhao_onibus"
    if re.search(padrao_moto, texto):
        return "moto"
    if re.search(padrao_veiculo_leve, texto):
        return "veiculo_leve"

    return "veiculo_generico"

## 5. Pipeline consolidado de tratamento

A função `tratar_bens` aplica as regras em ordem controlada e preserva a coluna `submacro_categoria` como nome auxiliar usado nas etapas seguintes.


In [75]:
def aplicar_regras_manuais(df, regras):
    """Aplica regras manuais definidas no topo do notebook."""
    df = df.copy()

    for regra in regras:
        nome = regra.get("nome", "regra_sem_nome")
        padrao = regra["padrao"]
        macro = regra.get("macro")
        submacro_col = regra.get("submacro_col")
        submacro_valor = regra.get("submacro_valor")

        mask = df["texto_norm"].str.contains(padrao, regex=True, na=False)

        if macro is not None:
            df.loc[mask, "macro_categoria_refinada"] = macro

        if submacro_col is not None and submacro_valor is not None:
            if submacro_col not in df.columns:
                df[submacro_col] = np.nan
            df.loc[mask, submacro_col] = submacro_valor

        df.loc[mask, "regra_manual_aplicada"] = (
            df.loc[mask, "regra_manual_aplicada"].fillna("").astype(str)
            + (";" + nome)
        ).str.strip(";")

    return df


def tratar_bens(df, params, regras_manuais=None):
    """Executa a versão final do tratamento dos bens declarados."""
    regras_manuais = regras_manuais or []

    df = preparar_base_inicial(df, params)
    df["regra_manual_aplicada"] = np.nan

    # 1. Imóveis
    df["submacro_imoveis"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_imoveis = df["macro_categoria"] == "imoveis"
    df.loc[mask_imoveis, "submacro_imoveis"] = (
        df.loc[mask_imoveis].apply(classificar_submacro_imoveis, axis=1)
    )
    df.loc[mask_imoveis & (df["submacro_imoveis"] == "imovel_rural"), "macro_categoria_refinada"] = "rural_agropecuario"
    df.loc[mask_imoveis & (df["submacro_imoveis"] == "participacoes_societarias"), "macro_categoria_refinada"] = "participacoes_societarias"

    # Mantém a coluna auxiliar de submacro de imóveis usada nas etapas seguintes.
    df["submacro_categoria"] = df["submacro_imoveis"]

    # 2. Ativos financeiros
    df["submacro_financeira"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_fin = df["macro_categoria_refinada"] == "ativos_financeiros"
    df.loc[mask_fin, "submacro_financeira"] = (
        df.loc[mask_fin].apply(classificar_submacro_financeira, axis=1)
    )
    df.loc[mask_fin & (df["submacro_financeira"] == "participacoes_societarias"), "macro_categoria_refinada"] = "participacoes_societarias"
    df.loc[mask_fin & (df["submacro_financeira"] == "rural_agropecuario"), "macro_categoria_refinada"] = "rural_agropecuario"

    # 3. Outros
    df["submacro_outros"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_outros = df["macro_categoria_refinada"] == "outros"
    df.loc[mask_outros, "submacro_outros"] = (
        df.loc[mask_outros].apply(classificar_submacro_outros, axis=1)
    )
    mask_outros_reclass = (
        (df["macro_categoria_refinada"] == "outros")
        & df["submacro_outros"].notna()
        & (df["submacro_outros"] != "outros")
    )
    df.loc[mask_outros_reclass, "macro_categoria_refinada"] = df.loc[mask_outros_reclass, "submacro_outros"]

    # 4. Veículos
    df["submacro_veiculos"] = pd.Series(pd.NA, index=df.index, dtype="object")
    mask_veiculos = df["macro_categoria_refinada"] == "veiculos"
    df.loc[mask_veiculos, "submacro_veiculos"] = (
        df.loc[mask_veiculos].apply(classificar_submacro_veiculos, axis=1)
    )

    # Correções adicionais para veículos genéricos.
    mask_veic_generico = (
        (df["macro_categoria_refinada"] == "veiculos")
        & (df["submacro_veiculos"] == "veiculo_generico")
    )
    texto_veic_generico = df["desc_norm"].astype(str)

    padrao_imovel_em_veiculos = r"\b(?:casa|alvenaria|rua|apartamento|terreno|lote|imovel|residencia|residencial|financiada pela caixa)\b"
    padrao_maquina_agricola_extra = r"\b(?:new holland|massey|ferguson|john deere|jonh deere|valtra|case ih|colheitadeira|plantadeira|maquinas e equipamentos|maquina agricola|trator|tratores|quadriciclo)\b"
    padrao_veiculo_leve_extra = r"\b(?:checrolet|chevrolet|corvette|m benz|benz|mercedes|trailblazer|traiblazer|c200|blindada|blindado|veiculos automotores|veiculo automotor|automotores|brp)\b"

    df.loc[mask_veic_generico & texto_veic_generico.str.contains(padrao_imovel_em_veiculos, regex=True, na=False), "submacro_veiculos"] = "imoveis"
    df.loc[mask_veic_generico & texto_veic_generico.str.contains(padrao_maquina_agricola_extra, regex=True, na=False), "submacro_veiculos"] = "maquina_agricola"
    df.loc[mask_veic_generico & texto_veic_generico.str.contains(padrao_veiculo_leve_extra, regex=True, na=False), "submacro_veiculos"] = "veiculo_leve"

    # Reclassificações finais vindas de veículos.
    mask_veiculos = df["macro_categoria_refinada"] == "veiculos"
    df.loc[mask_veiculos & (df["submacro_veiculos"] == "maquina_agricola"), "macro_categoria_refinada"] = "rural_agropecuario"
    df.loc[mask_veiculos & (df["submacro_veiculos"] == "imoveis"), "macro_categoria_refinada"] = "imoveis"

    # 5. Regras manuais finais
    if regras_manuais:
        df = aplicar_regras_manuais(df, regras_manuais)

    return df


## 6. Geração das features por candidato

Esta seção gera o dataset final de features patrimoniais: identificador do candidato, valores e percentuais por macro, métricas auxiliares, patrimônio total e UF.


In [76]:
def gerar_features_candidatos(bens_tratados, params):
    """
    Gera uma linha por candidato com as features finais do projeto.

    Saída:
    SQ_CANDIDATO, valores por macro, percentuais por macro,
    qtd_bens, qtd_macros_presentes, indice_concentracao_macro
    e patrimonio_total.
    """
    df = bens_tratados.copy()

    if "SQ_CANDIDATO" not in df.columns:
        raise KeyError("A base precisa conter SQ_CANDIDATO para gerar features por candidato.")

    macro_col = "macro_categoria_refinada"
    valor_col = "VR_BEM_CANDIDATO_NUM"
    macros_padrao = params.get("MACROS_FEATURES_PADRAO", [])

    df[macro_col] = df[macro_col].fillna(params.get("VALOR_PREENCHER_MACRO_NA", "outros"))

    valor_por_macro = (
        df.pivot_table(
            index="SQ_CANDIDATO",
            columns=macro_col,
            values=valor_col,
            aggfunc="sum",
            fill_value=0
        )
        .sort_index()
    )

    for macro in macros_padrao:
        if macro not in valor_por_macro.columns:
            valor_por_macro[macro] = 0

    if macros_padrao:
        valor_por_macro = valor_por_macro[macros_padrao]
    else:
        valor_por_macro = valor_por_macro.reindex(sorted(valor_por_macro.columns), axis=1)

    valor_por_macro.columns = [f"valor_{col}" for col in valor_por_macro.columns]
    colunas_valor_macro = list(valor_por_macro.columns)

    features = valor_por_macro.copy()
    features["patrimonio_total"] = features[colunas_valor_macro].sum(axis=1)

    colunas_percentuais = []
    for col in colunas_valor_macro:
        macro = col.replace("valor_", "", 1)
        perc_col = f"perc_{macro}"
        features[perc_col] = np.where(
            features["patrimonio_total"] > 0,
            features[col] / features["patrimonio_total"],
            0
        )
        colunas_percentuais.append(perc_col)

    features["qtd_bens"] = df.groupby("SQ_CANDIDATO").size()
    features["qtd_macros_presentes"] = (features[colunas_valor_macro] > 0).sum(axis=1)
    features["indice_concentracao_macro"] = (features[colunas_percentuais] ** 2).sum(axis=1)

    features = features.reset_index()

    ordem = (
        ["SQ_CANDIDATO"]
        + colunas_valor_macro
        + colunas_percentuais
        + [
            "qtd_bens",
            "qtd_macros_presentes",
            "indice_concentracao_macro",
            "patrimonio_total",
        ]
    )

    features = features[[col for col in ordem if col in features.columns]].copy()

    if params["REMOVER_CANDIDATOS_PATRIMONIO_ZERO_FEATURES"]:
        features = features[features["patrimonio_total"] > 0].copy()

    return features
def preparar_bens_modelagem(bens_tratados):
    """
    Gera a base de bens enxuta para auditoria: dados originais relevantes,
    valor numérico e macro final definida.
    """
    df = bens_tratados.copy()

    renomear = {}
    if "macro_categoria_refinada" in df.columns:
        renomear["macro_categoria_refinada"] = "macro_final"

    colunas_preferenciais = [
        "SQ_CANDIDATO",
        "NR_ORDEM_BEM_CANDIDATO",
        "DS_TIPO_BEM_CANDIDATO",
        "DS_BEM_CANDIDATO",
        "VR_BEM_CANDIDATO",
        "VR_BEM_CANDIDATO_NUM",
        "macro_categoria_refinada",
        "macro_final",
    ]

    colunas = [col for col in colunas_preferenciais if col in df.columns]
    df = df[colunas].rename(columns=renomear).copy()

    # Remove duplicações geradas quando a base já vier com nomes normalizados.
    df = df.loc[:, ~df.columns.duplicated()].copy()

    return df

## 7. Auditoria opcional da rodada

A auditoria é gerada somente quando `PARAMS["GERAR_AUDITORIA"] = True`.


In [77]:
def gerar_auditoria(bens_modelagem):
    """
    Gera a auditoria  da rodada.
    """
    return {
        "bens_original_com_macro_final": bens_modelagem.copy()
    }


## 8. Exportação dos resultados

Cada rodada gera uma pasta própria com o dataset final. A auditoria, quando ativada, é exportada em pasta separada por rodada.


In [78]:
def exportar_resultados(bens_modelagem, features_candidatos, auditoria, params):
    """Salva o dataset final e, opcionalmente, a auditoria enxuta da rodada."""
    dir_saida = Path(params["DIR_SAIDA_BASE"]) / params["NOME_RODADA"]
    dir_saida.mkdir(parents=True, exist_ok=True)

    sep = params["SEP_SAIDA"]
    enc = params["ENCODING_SAIDA"]

    caminhos = {}

    caminhos["features_candidatos"] = dir_saida / "features_patrimoniais_por_candidato.csv"
    features_candidatos.to_csv(caminhos["features_candidatos"], sep=sep, index=False, encoding=enc)

    if params.get("GERAR_AUDITORIA", False):
        dir_auditoria = Path(params["DIR_SAIDA_BASE"]) / "auditoria" / params["NOME_RODADA"]
        dir_auditoria.mkdir(parents=True, exist_ok=True)

        caminhos["auditoria_bens_original_com_macro_final"] = (
            dir_auditoria / "bens_original_com_macro_final.csv"
        )
        auditoria["bens_original_com_macro_final"].to_csv(
            caminhos["auditoria_bens_original_com_macro_final"],
            sep=sep,
            index=False,
            encoding=enc
        )

    if params.get("GERAR_CONFIG", True):
        config_export = {
            "PARAMS": params,
            "AJUSTES_TIPO_MACRO": AJUSTES_TIPO_MACRO,
            "REGRAS_MANUAIS_TEXTO": REGRAS_MANUAIS_TEXTO,
        }

        caminhos["config"] = dir_saida / "config_rodada.json"
        with open(caminhos["config"], "w", encoding="utf-8") as f:
            json.dump(config_export, f, ensure_ascii=False, indent=2)

    if params.get("GERAR_ZIP_RODADA", False):
        zip_path = Path(params["DIR_SAIDA_BASE"]) / f"{params['NOME_RODADA']}.zip"
        if zip_path.exists():
            zip_path.unlink()
        shutil.make_archive(str(zip_path).replace(".zip", ""), "zip", dir_saida)
        caminhos["zip_rodada"] = zip_path

    return caminhos


def executar_pipeline(params, regras_manuais=None):
    """Executa a rodada completa do pipeline."""
    df_raw, origem = ler_base_bens(params)
    bens_tratados = tratar_bens(df_raw, params, regras_manuais=regras_manuais)
    bens_modelagem = preparar_bens_modelagem(bens_tratados)
    features_candidatos = gerar_features_candidatos(bens_tratados, params)
    auditoria = gerar_auditoria(bens_modelagem) if params.get("GERAR_AUDITORIA", False) else {}
    caminhos = exportar_resultados(bens_modelagem, features_candidatos, auditoria, params)

    return {
        "origem": origem,
        "bens_tratados": bens_tratados,
        "bens_modelagem": bens_modelagem,
        "features_candidatos": features_candidatos,
        "auditoria": auditoria,
        "caminhos": caminhos,
    }


## 9. Execução e exportação

Execute esta célula para rodar o tratamento completo. Para ativar a auditoria, altere `GERAR_AUDITORIA` nos parâmetros.


In [79]:
resultado = executar_pipeline(
    PARAMS,
    regras_manuais=REGRAS_MANUAIS_TEXTO
)

print("Origem:", resultado["origem"])
print("Bens tratados:", resultado["bens_tratados"].shape)
print("Bens para auditoria:", resultado["bens_modelagem"].shape)
print("Features por candidato:", resultado["features_candidatos"].shape)
print("\nArquivos gerados:")
for nome, caminho in resultado["caminhos"].items():
    print(f"- {nome}: {caminho}")


Origem: arquivo_unificado:bem_candidato_2022_BRASIL.csv | encoding=latin1 | sep=;
Bens tratados: (92538, 32)
Bens para auditoria: (92538, 7)
Features por candidato: (18219, 27)

Arquivos gerados:
- features_candidatos: features_patrimoniais_por_candidato/features_patrimoniais_por_candidato.csv
- config: features_patrimoniais_por_candidato/config_rodada.json


## 10. Revisão rápida da rodada

Use as tabelas abaixo para conferir a estrutura final gerada pelo pipeline.


In [80]:
resultado["features_candidatos"].head()


,SQ_CANDIDATO,valor_ativos_financeiros,valor_bens_luxo_colecao,valor_creditos_direitos,valor_dinheiro_especie,valor_direitos_intangiveis,valor_imoveis,valor_outros,valor_outros_atividade_profissional,valor_participacoes_societarias,valor_rural_agropecuario,valor_veiculos,perc_ativos_financeiros,perc_bens_luxo_colecao,perc_creditos_direitos,perc_dinheiro_especie,perc_direitos_intangiveis,perc_imoveis,perc_outros,perc_outros_atividade_profissional,perc_participacoes_societarias,perc_rural_agropecuario,perc_veiculos,qtd_bens,qtd_macros_presentes,indice_concentracao_macro,patrimonio_total
0,10001595335,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"60,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,2,1,1.00,"60,000.00"
1,10001595336,0.00,0.00,0.00,0.00,0.00,"400,000.00",0.00,0.00,0.00,0.00,"113,900.00",0.00,0.00,0.00,0.00,0.00,0.78,0.00,0.00,0.00,0.00,0.22,3,2,0.65,"513,900.00"
2,10001595338,0.00,0.00,0.00,0.00,0.00,"300,000.00",0.00,0.00,0.00,0.00,"12,000.00",0.00,0.00,0.00,0.00,0.00,0.96,0.00,0.00,0.00,0.00,0.04,2,2,0.93,"312,000.00"
3,10001595339,0.00,0.00,0.00,0.00,0.00,"250,000.00",0.00,0.00,0.00,0.00,"100,000.00",0.00,0.00,0.00,0.00,0.00,0.71,0.00,0.00,0.00,0.00,0.29,2,2,0.59,"350,000.00"
4,10001595340,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"32,000.00",0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,0.00,1.00,2,1,1.00,"32,000.00"


In [81]:
resultado["features_candidatos"].columns.tolist()


['SQ_CANDIDATO',
 'valor_ativos_financeiros',
 'valor_bens_luxo_colecao',
 'valor_creditos_direitos',
 'valor_dinheiro_especie',
 'valor_direitos_intangiveis',
 'valor_imoveis',
 'valor_outros',
 'valor_outros_atividade_profissional',
 'valor_participacoes_societarias',
 'valor_rural_agropecuario',
 'valor_veiculos',
 'perc_ativos_financeiros',
 'perc_bens_luxo_colecao',
 'perc_creditos_direitos',
 'perc_dinheiro_especie',
 'perc_direitos_intangiveis',
 'perc_imoveis',
 'perc_outros',
 'perc_outros_atividade_profissional',
 'perc_participacoes_societarias',
 'perc_rural_agropecuario',
 'perc_veiculos',
 'qtd_bens',
 'qtd_macros_presentes',
 'indice_concentracao_macro',
 'patrimonio_total']

In [82]:
resultado["bens_modelagem"].head()


,SQ_CANDIDATO,NR_ORDEM_BEM_CANDIDATO,DS_TIPO_BEM_CANDIDATO,DS_BEM_CANDIDATO,VR_BEM_CANDIDATO,VR_BEM_CANDIDATO_NUM,macro_final
0,20001652525,13,OUTROS BENS E DIREITOS,"PARTICIPACAO NA SOCIEDADE HOSPITAL SANTA INES,...","10,000.00","10,000.00",participacoes_societarias
1,20001652525,14,Dinheiro em espécie - moeda nacional,DINHEIRO EM ESPECIE,"30,000.00","30,000.00",dinheiro_especie
2,20001652525,15,OUTROS BENS E DIREITOS,CAMARA MUNICIPAL DE CORURIPE,"104,000.00","104,000.00",outros
3,20001652526,1,OUTROS BENS E DIREITOS,RENDIMENTO ANUAL ALAGOAS CAMARA DOS VEREADORES,"66,080.00","66,080.00",outros
4,20001652527,1,OUTROS BENS E DIREITOS,CONTA CORRENTE,993.66,993.66,ativos_financeiros


In [83]:
resultado["features_candidatos"][["qtd_bens", "qtd_macros_presentes", "indice_concentracao_macro", "patrimonio_total"]].describe().T


,count,mean,std,min,25%,50%,75%,max
qtd_bens,"18,219.00",5.08,7.12,1.00,1.00,3.00,6.00,180.00
qtd_macros_presentes,"18,219.00",2.36,1.39,1.00,1.00,2.00,3.00,9.00
indice_concentracao_macro,"18,219.00",0.76,0.24,0.18,0.54,0.80,1.00,1.00
patrimonio_total,"18,219.00","1,299,809.44","13,692,840.46",0.01,"78,000.00","277,000.00","758,591.17","1,267,950,846.18"
